# 02 - Self-Training Walkthrough

This notebook demonstrates each component of the self-training pipeline with interactive, runnable code. We use toy models and synthetic data to make every concept concrete and executable.

**What you will learn:**
1. What is self-training? (conceptual overview)
2. How EMA teacher works (with parameter trajectory plots)
3. Confidence thresholding for pseudo-label filtering
4. Curriculum scheduling across rounds
5. The combined loss function

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

%matplotlib inline

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
torch.manual_seed(42)
rng = np.random.RandomState(42)

## 1. What Is Self-Training?

Self-training is a semi-supervised learning technique where a model generates labels for unlabeled data, then retrains on the expanded dataset. The process repeats iteratively.

```
Standard Training:                Self-Training:

  Labeled Data ----> Model         Labeled Data ----+
                                                     |---> Model ---> Pseudo-Labels
  Unlabeled Data     (unused)      Unlabeled Data ---+         |            |
                                                               |            |
                                                               +<--- Retrain with both
                                                                     (repeat N rounds)
```

### The Confirmation Bias Problem

Naive self-training has a critical flaw: **the model reinforces its own errors.** If the model confidently mislabels a region, it will train on that mistake and become even more confident in it. This is called confirmation bias.

Our pipeline addresses this with three defenses:
1. **EMA teacher**: A smoothed version of the model that provides more stable pseudo-labels
2. **Confidence filtering**: Only keep pseudo-labels where the model is highly confident
3. **Curriculum thresholding**: Start strict, relax gradually as the model improves

## 2. EMA Teacher: Concept and Demo

The Exponential Moving Average (EMA) teacher maintains a running average of the student's parameters:

```
theta_teacher = alpha * theta_teacher + (1 - alpha) * theta_student
```

With alpha = 0.999, the teacher is a weighted average of the last ~1000 student checkpoints. This acts as an implicit **temporal ensemble**, smoothing out noisy gradient steps.

Let's visualize this with a simple 1D example.

In [ ]:
# Simple toy model: a single linear layer
class ToyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1, bias=False)
        nn.init.constant_(self.linear.weight, 0.5)  # Start at w=0.5

    def forward(self, x):
        return self.linear(x)


def simulate_ema(n_steps=200, ema_decay=0.999, lr=0.01, target=2.0):
    """Simulate student SGD training and EMA teacher tracking."""
    student = ToyModel()
    # Teacher starts as a copy of the student
    teacher_weight = student.linear.weight.data.item()

    student_history = [student.linear.weight.data.item()]
    teacher_history = [teacher_weight]

    optimizer = torch.optim.SGD(student.parameters(), lr=lr)

    for step in range(n_steps):
        # Forward pass: student tries to learn f(x) = target * x
        x = torch.randn(16, 1)
        y_true = target * x + 0.3 * torch.randn(16, 1)  # Noisy target
        y_pred = student(x)

        loss = nn.MSELoss()(y_pred, y_true)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # EMA update
        student_w = student.linear.weight.data.item()
        teacher_weight = ema_decay * teacher_weight + (1 - ema_decay) * student_w

        student_history.append(student_w)
        teacher_history.append(teacher_weight)

    return student_history, teacher_history


# Run simulation
student_hist, teacher_hist = simulate_ema(n_steps=300, ema_decay=0.999)

fig, ax = plt.subplots(figsize=(12, 5))
steps = range(len(student_hist))
ax.plot(steps, student_hist, alpha=0.6, linewidth=1, label="Student (SGD)", color="#3498db")
ax.plot(steps, teacher_hist, linewidth=2.5, label="Teacher (EMA, alpha=0.999)", color="#e74c3c")
ax.axhline(y=2.0, color="black", linestyle="--", alpha=0.5, label="Target (w=2.0)")
ax.set_xlabel("Training Step")
ax.set_ylabel("Weight Value")
ax.set_title("EMA Teacher vs Student Parameter Trajectories", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final student weight: {student_hist[-1]:.4f}")
print(f"Final teacher weight: {teacher_hist[-1]:.4f}")
print(f"Target weight:        2.0000")
print()
print("Observation: The teacher follows the student but with much less noise.")
print("This smoothing makes the teacher better for generating pseudo-labels.")

In [ ]:
# Compare different EMA decay rates
decay_rates = [0.9, 0.99, 0.999, 0.9999]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, decay in enumerate(decay_rates):
    student_h, teacher_h = simulate_ema(n_steps=300, ema_decay=decay)
    axes[i].plot(range(len(student_h)), student_h, alpha=0.4, linewidth=0.8, color="#3498db", label="Student")
    axes[i].plot(range(len(teacher_h)), teacher_h, linewidth=2, color="#e74c3c", label="Teacher")
    axes[i].axhline(y=2.0, color="black", linestyle="--", alpha=0.4)
    axes[i].set_title(f"alpha = {decay}", fontsize=11, fontweight="bold")
    axes[i].set_xlabel("Step")
    axes[i].set_ylabel("Weight")
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.2)

fig.suptitle("Effect of EMA Decay Rate on Teacher Smoothness", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("alpha = 0.9:    Very responsive, almost tracks the student (noisy)")
print("alpha = 0.99:   Moderate smoothing, adapts within ~100 steps")
print("alpha = 0.999:  Strong smoothing, adapts within ~1000 steps [OUR CHOICE]")
print("alpha = 0.9999: Very slow adaptation, may lag too much")

## 3. Confidence Thresholding

Not all pseudo-labels are trustworthy. We filter by confidence: only voxels where the model's prediction probability exceeds a threshold are used for training.

For a sigmoid output p in [0, 1]:
- Confidence = max(p, 1-p)
- If confidence >= threshold, the pseudo-label is accepted
- If confidence < threshold, the voxel is masked out

In [ ]:
# Simulate model predictions on a 2D grid
size = 50
y_grid, x_grid = np.mgrid[-3:3:complex(size), -3:3:complex(size)]

# True boundary: a circle
true_label = ((x_grid - 0.3) ** 2 + (y_grid + 0.2) ** 2 < 2.0).astype(np.float32)

# Simulated model output (sigmoid probabilities)
# Higher confidence in the center, lower near boundaries
distance_to_boundary = np.abs(np.sqrt((x_grid - 0.3) ** 2 + (y_grid + 0.2) ** 2) - np.sqrt(2.0))
noise = rng.normal(0, 0.3, (size, size))
raw_logits = 3.0 * (true_label - 0.5) + 1.5 * distance_to_boundary * (2 * true_label - 1) + noise
model_prob = 1.0 / (1.0 + np.exp(-raw_logits))  # Sigmoid

# Confidence = max(p, 1-p)
confidence = np.maximum(model_prob, 1 - model_prob)

# Show filtering at different thresholds
thresholds = [0.75, 0.85, 0.95]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Ground truth
axes[0].imshow(true_label, cmap="RdYlGn", vmin=0, vmax=1)
axes[0].set_title("Ground Truth", fontsize=11)
axes[0].axis("off")

for i, thresh in enumerate(thresholds):
    mask = confidence >= thresh
    accepted_pct = 100 * mask.sum() / mask.size

    # Show pseudo-label with mask
    pseudo = (model_prob > 0.5).astype(np.float32)
    display = np.ones((size, size, 3)) * 0.85  # Gray background for rejected
    display[mask & (pseudo == 1)] = [0.2, 0.7, 0.3]  # Green for accepted positive
    display[mask & (pseudo == 0)] = [0.9, 0.3, 0.3]  # Red for accepted negative

    axes[i + 1].imshow(display)
    axes[i + 1].set_title(f"Threshold = {thresh}\n({accepted_pct:.0f}% accepted)", fontsize=10)
    axes[i + 1].axis("off")

fig.suptitle(
    "Confidence Filtering: Higher Thresholds Accept Fewer but Cleaner Pseudo-Labels",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print("Gray = rejected (below threshold) | Green = accepted positive | Red = accepted negative")
print()
print("Notice: boundary regions are rejected first (lowest confidence).")
print("Higher thresholds keep only the 'easy' interior/exterior voxels.")

## 4. Curriculum Threshold Scheduling

A fixed threshold faces a dilemma:
- **Too high** (0.95): Very few pseudo-labels accepted, wasting unlabeled data
- **Too low** (0.75): Many noisy pseudo-labels, risking confirmation bias

**Curriculum thresholding** resolves this: start strict, relax as the model improves.

```
tau(r) = tau_final + (tau_initial - tau_final) * 0.5 * (1 + cos(pi * r / (R-1)))
```

In [ ]:
import math


def compute_threshold(round_num, num_rounds=4, initial=0.95, final=0.75, schedule="cosine"):
    """Compute threshold for a given round."""
    if num_rounds == 1:
        return initial
    t = round_num / (num_rounds - 1)
    if schedule == "linear":
        return initial + (final - initial) * t
    if schedule == "cosine":
        return final + (initial - final) * 0.5 * (1.0 + math.cos(math.pi * t))
    # step
    return initial if t < 0.5 else final


# Plot all three schedule types
num_rounds = 4
rounds = np.linspace(0, num_rounds - 1, 100)
schedules = ["linear", "cosine", "step"]
colors_sched = ["#3498db", "#e74c3c", "#2ecc71"]

fig, ax = plt.subplots(figsize=(10, 5))

for schedule, color in zip(schedules, colors_sched):
    thresholds_curve = [compute_threshold(r, num_rounds, schedule=schedule) for r in rounds]
    ax.plot(rounds, thresholds_curve, linewidth=2.5, label=f"{schedule.capitalize()} schedule", color=color)

# Mark actual round values for cosine
for r in range(num_rounds):
    t = compute_threshold(r, num_rounds, schedule="cosine")
    ax.plot(r, t, "ko", markersize=8)
    ax.annotate(f"  R{r}: {t:.3f}", (r, t), fontsize=9)

ax.set_xlabel("Self-Training Round", fontsize=12)
ax.set_ylabel("Confidence Threshold", fontsize=12)
ax.set_title("Curriculum Threshold Schedules", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(0.7, 1.0)
ax.set_xticks(range(num_rounds))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class threshold visualization
class_names = ["TC", "WT", "ET"]
offsets = {"TC": 0.0, "WT": -0.05, "ET": 0.05}
class_colors = {"TC": "#e74c3c", "WT": "#27ae60", "ET": "#f39c12"}

fig, ax = plt.subplots(figsize=(10, 5))

for cls in class_names:
    thresholds_cls = []
    for r in range(num_rounds):
        global_t = compute_threshold(r, num_rounds, schedule="cosine")
        class_t = min(max(global_t + offsets[cls], 0.5), 1.0)
        thresholds_cls.append(class_t)
    ax.plot(
        range(num_rounds),
        thresholds_cls,
        "o-",
        linewidth=2.5,
        markersize=10,
        label=f"{cls} (offset={offsets[cls]:+.2f})",
        color=class_colors[cls],
    )
    for r, t in enumerate(thresholds_cls):
        ax.annotate(f" {t:.2f}", (r, t), fontsize=8, va="bottom")

ax.set_xlabel("Self-Training Round", fontsize=12)
ax.set_ylabel("Confidence Threshold", fontsize=12)
ax.set_title("Per-Class Curriculum Thresholds (Cosine Schedule)", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xticks(range(num_rounds))
ax.set_xticklabels([f"Round {r}" for r in range(num_rounds)])
ax.set_ylim(0.65, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  - ET starts at 1.00 (clamped): only perfect predictions accepted in Round 0")
print("  - WT starts at 0.90: more lenient because WT is the easiest class")
print("  - By Round 3, all classes accept pseudo-labels at 0.70-0.80 confidence")

## 5. Combined Loss Function

The self-training loss has three components:

```
L_total = w_sup * L_supervised  +  ramp(e) * w_pseudo * L_pseudo  +  ramp(e) * w_consist * L_consistency
```

| Component | What it does | Weight |
|-----------|-------------|--------|
| Supervised (DiceCE) | Learns from real labels | 1.0 |
| Pseudo-label (DiceCE) | Learns from teacher's pseudo-labels | 0.5 |
| Consistency (MSE) | Student-teacher agreement | 0.1 |

The ramp-up function prevents the model from being overwhelmed by potentially noisy pseudo-labels at the start of each round.

In [ ]:
# Visualize the ramp-up schedule
ramp_up_epochs = 20
epochs = np.arange(0, 100)
ramp_weights = np.minimum(epochs / ramp_up_epochs, 1.0)

# Effective weights over time
w_sup = 1.0
w_pseudo = 0.5
w_consist = 0.1

eff_supervised = np.full_like(epochs, w_sup, dtype=float)
eff_pseudo = w_pseudo * ramp_weights
eff_consistency = w_consist * ramp_weights

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Ramp function
ax1.plot(epochs, ramp_weights, linewidth=2.5, color="#8e44ad")
ax1.axvline(x=ramp_up_epochs, color="gray", linestyle="--", alpha=0.5, label=f"Ramp-up complete (epoch {ramp_up_epochs})")
ax1.set_xlabel("Epoch (within round)")
ax1.set_ylabel("Ramp Weight")
ax1.set_title("Linear Ramp-Up Schedule", fontsize=12, fontweight="bold")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-0.05, 1.1)

# Right: Effective loss weights
ax2.fill_between(epochs, 0, eff_supervised, alpha=0.3, label=f"Supervised (w={w_sup})", color="#3498db")
ax2.fill_between(epochs, eff_supervised, eff_supervised + eff_pseudo, alpha=0.3, label=f"Pseudo-label (w={w_pseudo})", color="#e74c3c")
ax2.fill_between(epochs, eff_supervised + eff_pseudo, eff_supervised + eff_pseudo + eff_consistency, alpha=0.3, label=f"Consistency (w={w_consist})", color="#2ecc71")

ax2.plot(epochs, eff_supervised, linewidth=2, color="#3498db")
ax2.plot(epochs, eff_supervised + eff_pseudo, linewidth=2, color="#e74c3c")
ax2.plot(epochs, eff_supervised + eff_pseudo + eff_consistency, linewidth=2, color="#2ecc71")

ax2.set_xlabel("Epoch (within round)")
ax2.set_ylabel("Effective Loss Weight")
ax2.set_title("Loss Component Weights Over Training", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("At epoch 0:  Only supervised loss is active (ramp=0.0)")
print(f"At epoch {ramp_up_epochs}: All components reach full weight (ramp=1.0)")
print("This prevents early training instability from noisy pseudo-labels.")

In [ ]:
# Demonstrate the combined loss with toy tensors
# (Uses actual project code if available, otherwise pure PyTorch)

batch_size = 2
num_classes = 3
spatial = (8, 8, 8)  # Small spatial dims for demo

# Simulate student logits, labels, pseudo-labels, teacher logits
student_logits = torch.randn(batch_size, num_classes, *spatial)
labels = torch.randint(0, 2, (batch_size, num_classes, *spatial)).float()
pseudo_labels = torch.sigmoid(torch.randn(batch_size, num_classes, *spatial))
teacher_logits = torch.randn(batch_size, num_classes, *spatial)
confidence_mask = (torch.rand(batch_size, 1, *spatial) > 0.3).float()  # 70% confident

# Compute individual loss components
mse_loss = nn.MSELoss()

# Simplified DiceCE for demo
def simple_dice_loss(pred, target, smooth=1e-6):
    pred_prob = torch.sigmoid(pred)
    intersection = (pred_prob * target).sum()
    return 1.0 - (2.0 * intersection + smooth) / (pred_prob.sum() + target.sum() + smooth)


# Compute each component
l_sup = simple_dice_loss(student_logits, labels)
l_pseudo = simple_dice_loss(student_logits * confidence_mask, pseudo_labels * confidence_mask)
l_consist = mse_loss(student_logits, teacher_logits.detach())

# Combined with ramp weight
epoch = 10
ramp = min(epoch / 20, 1.0)
l_total = w_sup * l_sup + ramp * w_pseudo * l_pseudo + ramp * w_consist * l_consist

print("Loss Components (toy example):")
print(f"  Supervised loss:    {l_sup.item():.4f}  (weight: {w_sup})")
print(f"  Pseudo-label loss:  {l_pseudo.item():.4f}  (weight: {w_pseudo}, ramp: {ramp:.2f})")
print(f"  Consistency loss:   {l_consist.item():.4f}  (weight: {w_consist}, ramp: {ramp:.2f})")
print(f"  ---")
print(f"  Total loss:         {l_total.item():.4f}")
print(f"")
print(f"At epoch {epoch}/{ramp_up_epochs}, ramp weight = {ramp:.2f}")
print(f"Pseudo and consistency losses contribute {ramp*100:.0f}% of their full weight.")

## 6. Putting It All Together: The Self-Training Loop

Here is the complete pipeline for one round of self-training:

```
For round r in {0, 1, 2, 3}:

    1. THRESHOLD: Get per-class thresholds from curriculum scheduler
       tau_TC, tau_WT, tau_ET = scheduler.get_thresholds(r)

    2. PSEUDO-LABEL: Teacher generates labels for 266 unlabeled volumes
       For each unlabeled volume:
           pred = sigmoid(teacher(volume))     # Sliding window inference
           conf = max(pred, 1-pred)            # Per-voxel confidence
           mask = conf >= threshold             # Per-class filtering
           if enough_confident_voxels(mask):    # Min 30% per class
               save_pseudo_label(pred, mask)

    3. DATASET: Combine labeled + accepted pseudo-labeled volumes
       dataset = D_train + D_pseudo_accepted

    4. TRAIN: 100 epochs with combined loss
       For each epoch:
           For each batch:
               student_pred = student(strong_aug(input))
               teacher_pred = teacher(weak_aug(input))
               loss = combined_loss(student_pred, labels_or_pseudo, teacher_pred)
               loss.backward()
               optimizer.step()
               teacher.ema_update(student)      # After each step
           Validate every 5 epochs on D_val

    5. CHECKPOINT: Save round r state
```

**Next notebook:** [03_results_comparison.ipynb](03_results_comparison.ipynb) analyzes the results of training.